In [1]:
# PySpark is the Spark API for Python. In this project, I use PySpark to initialize the SparkContext.   

from pyspark import SparkContext, SparkConf

from pyspark.sql import SparkSession
from pyspark.sql.functions import sum, lit, avg,when, to_date, year,quarter

In [2]:
# importing and initializing findspark for easy spark setup.
import findspark

findspark.init()

The data sources were downloaded via terminal to my working directory.

curl -o dataset1.csv 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-BD0225EN-SkillsNetwork/labs/data/dataset1.csv'


curl -o dataset2.csv 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-BD0225EN-SkillsNetwork/labs/data/dataset2.csv'

In [3]:
## Setting the environment variable to reference the java path

import os

#declaring the java environment variable
os.environ["JAVA_HOME"] = "C:/Program Files/Java/jdk-22"

# Creating a SparkContext object

sc = SparkContext.getOrCreate()

# Creating a Spark Session

spark = SparkSession \
    .builder \
    .appName("Python Spark DataFrames basic example") \
    .getOrCreate()


##When I executed my code without stating below configuration, setting timeparse policy, an exception was encountred while executing df.show() in cell [9]. Adding this config solved the issue.
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")


In [4]:
### checking the spark properties 
spark

In [5]:
# Loading the data into spark dataframe

df1 = spark.read.csv("C:\\Users\\Gbenga\\OneDrive\\Documents\\Pyspark_Project\\dataset1.csv", header=True, inferSchema= True)

df2 = spark.read.csv("C:\\Users\\Gbenga\\OneDrive\\Documents\\Pyspark_Project\\dataset2.csv", header=True, inferSchema= True)

In [6]:
# viewing data frames schema
df1.printSchema()

df2.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- date_column: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- description: string (nullable = true)
 |-- location: string (nullable = true)

root
 |-- customer_id: integer (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- value: integer (nullable = true)
 |-- notes: string (nullable = true)



In [7]:
#cleaning the dataframes by dropping some columns

df1= df1.drop('description','location')
df2 = df2.drop('notes')

In [8]:
#Add new column year to df1 by extracting year from date column
df1= df1.withColumn('year', year(to_date('date_column','dd/MM/yyyy')))

#Add new column quarter to df2 by extracting quarter from transaction date column

df2= df2.withColumn('quarter', quarter(to_date('transaction_date','dd/MM/yyyy')))



In [9]:
# Joining the two dataframes together on customer id fields

df = df1.join(df2, on= 'customer_id', how= 'inner' )


# renaming amount and value columns
df = df.withColumnRenamed('amount','transaction_amount')
df = df.withColumnRenamed('value','transaction_value')

df.show()

+-----------+-----------+------------------+----+----------------+-----------------+-------+
|customer_id|date_column|transaction_amount|year|transaction_date|transaction_value|quarter|
+-----------+-----------+------------------+----+----------------+-----------------+-------+
|          1|   1/1/2022|              5000|2022|      01/01/2022|             1500|      1|
|          2|  15/2/2022|              1200|2022|      15/02/2022|             2000|      1|
|          3|  20/3/2022|               800|2022|      20/03/2022|             1000|      1|
|          4|  10/4/2022|              3000|2022|      10/04/2022|             2500|      2|
|          5|   5/5/2022|              6000|2022|      05/05/2022|             1800|      2|
|          6|  10/6/2022|              4500|2022|      10/06/2022|             1200|      2|
|          7|  15/7/2022|               200|2022|      15/07/2022|              700|      3|
|          8|  20/8/2022|              3500|2022|      20/08/2022|    

In [10]:
## Creating a temporary view called Transactions to allow spark sql operations.
df.createTempView("Transactions")

In [11]:
## Filtering the merged data by transaction amount greater than 1500.
df_filter = spark.sql("select * from transactions where transaction_amount > 1500")

df_filter.show()

+-----------+-----------+------------------+----+----------------+-----------------+-------+
|customer_id|date_column|transaction_amount|year|transaction_date|transaction_value|quarter|
+-----------+-----------+------------------+----+----------------+-----------------+-------+
|          1|   1/1/2022|              5000|2022|      01/01/2022|             1500|      1|
|          4|  10/4/2022|              3000|2022|      10/04/2022|             2500|      2|
|          5|   5/5/2022|              6000|2022|      05/05/2022|             1800|      2|
|          6|  10/6/2022|              4500|2022|      10/06/2022|             1200|      2|
|          8|  20/8/2022|              3500|2022|      20/08/2022|             3000|      3|
|         10| 30/10/2022|              1800|2022|      30/10/2022|             1200|      4|
|         11|  5/11/2022|              2200|2022|       5/11/2022|             1500|      4|
|         13|  15/1/2023|              4800|2023|      15/01/2023|    

In [16]:
## Getting the maximum transaction amount in each sales quarter.
max_quarter = spark.sql("select quarter,max(transaction_amount) from transactions group by quarter")

max_quarter.show()

+-------+-----------------------+
|quarter|max(transaction_amount)|
+-------+-----------------------+
|      1|                   5000|
|      3|                   5500|
|      4|                   2400|
|      2|                   6000|
+-------+-----------------------+



In [12]:
## Agrregation total sales by year
grouped_agg = df.groupBy('year').agg(sum('transaction_amount').alias('total_sales')).sort('year')

grouped_agg.show()

##grouped_agg.sort('total_sales').show()

+----+-----------+
|year|total_sales|
+----+-----------+
|2022|      29800|
|2023|      28100|
|2024|      25700|
|2025|      25700|
|2026|      25700|
|2027|      25700|
|2028|      25700|
|2029|      25700|
|2030|       9500|
+----+-----------+



In [13]:
## adding a column based on benchmark of transaction amount greater than 5000

df=df.withColumn('benchmark', when(df['transaction_amount'] > 5000, lit('Yes')).otherwise(lit('No')))

df.show()

+-----------+-----------+------------------+----+----------------+-----------------+-------+---------+
|customer_id|date_column|transaction_amount|year|transaction_date|transaction_value|quarter|benchmark|
+-----------+-----------+------------------+----+----------------+-----------------+-------+---------+
|          1|   1/1/2022|              5000|2022|      01/01/2022|             1500|      1|       No|
|          2|  15/2/2022|              1200|2022|      15/02/2022|             2000|      1|       No|
|          3|  20/3/2022|               800|2022|      20/03/2022|             1000|      1|       No|
|          4|  10/4/2022|              3000|2022|      10/04/2022|             2500|      2|       No|
|          5|   5/5/2022|              6000|2022|      05/05/2022|             1800|      2|      Yes|
|          6|  10/6/2022|              4500|2022|      10/06/2022|             1200|      2|       No|
|          7|  15/7/2022|               200|2022|      15/07/2022|       

In [ ]:
## Average transaction amount per customer

customer_transactions = df.groupBy('customer_id').agg(avg('transaction_amount').alias('average_trans'))

## Writing the average customer transaction amount to hive table

customer_transactions.write.mode("overwrite").saveAsTable('customer_avg')

In [ ]:
#Write aggregated customer data to HDFS in parquet format file filtered_data.parquet

customer_transactions.write.mode("overwrite").parquet("customer_transactions.parquet")

Thank You for the follow up!!!